# %% [markdown]

 # Grocery Recommendations Prototype – Task 1 & 2

 This notebook covers:
 1. Mining frequent itemsets (Task 1)
 2. Building a baseline collaborative‑filtering recommender (Task 2)

 **Notebook style guide**
 * No large monolithic functions – each step is explicit.
 * Run cells top‑to‑bottom.
 * Adjust paths / parameters as needed.

 ---


# %% [markdown]

 ## 0 Imports & dataset paths


In [23]:
# %%

import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import warnings, datetime as dt, math, itertools
warnings.filterwarnings('ignore')

# %% [markdown]

 ## 1 Load train / test splits


In [24]:
# %%

train_path = 'Groceries data train.csv'
test_path  = 'Groceries data test.csv'
train_raw  = pd.read_csv(train_path)
test_raw   = pd.read_csv(test_path)

# Debug prints
print("Train columns before strip:", train_raw.columns.tolist())
print("Test columns before strip:", test_raw.columns.tolist())

# Clean column names by stripping whitespace
train_raw.columns = train_raw.columns.str.strip()
test_raw.columns = test_raw.columns.str.strip()

# Debug prints after cleaning
print("\nTrain columns after strip:", train_raw.columns.tolist())
print("Test columns after strip:", test_raw.columns.tolist())

print("\nFirst few rows of train data:")
print(train_raw.head())
print("\nFirst few rows of test data:")
print(test_raw.head())

# Standardize column names
test_raw = test_raw.rename(columns={'user_id': 'User_id'})

# Handle any NA values in User_id
train_raw['User_id'] = train_raw['User_id'].fillna(0)
test_raw['User_id'] = test_raw['User_id'].fillna(0)

train_raw['BasketID'] = train_raw['User_id'].astype(int).astype(str) + '_' + train_raw['Date']
test_raw['BasketID']  = test_raw['User_id'].astype(int).astype(str) + '_' + test_raw['Date']

print('Unique baskets in train:', train_raw['BasketID'].nunique())

Train columns before strip: ['User_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']
Test columns before strip: ['user_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']

Train columns after strip: ['User_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']
Test columns after strip: ['user_id', 'Date', 'itemDescription', 'year', 'month', 'day', 'day_of_week']

First few rows of train data:
   User_id       Date itemDescription  year  month  day  day_of_week
0     2351  1/01/2014         cleaner  2014      1    1            2
1     2226  1/01/2014         sausage  2014      1    1            2
2     1922  1/01/2014  tropical fruit  2014      1    1            2
3     2943  1/01/2014      whole milk  2014      1    1            2
4     1249  1/01/2014    citrus fruit  2014      1    1            2

First few rows of test data:
   user_id        Date   itemDescription  year  month  day  day_of_week
0     2889  20/01/2015          

# %% [markdown]

 ## 4 One‑hot encode baskets for Apriori


In [25]:
# # %%

# basket_train = (
#     train_raw
#       .groupby(['BasketID', 'itemDescription'])['itemDescription']
#       .count()
#       .unstack(fill_value=0)
#       .astype(bool)
#       .astype(int)
# )

# print('Basket × item matrix shape:', basket_train.shape)
# basket_train.head()

# %% [markdown]

 ## 5 Frequent itemset mining (Apriori)
 * **min_support** set low (e.g. 0.01) – adjust to taste.
 * We keep itemsets up to length 3 to keep things tractable.


In [26]:
# # %%

# min_support = 0.02
# freq_itemsets = apriori(basket_train,
#                         min_support=min_support,
#                         use_colnames=True,
#                         max_len=3,
#                         low_memory=True)

# freq_itemsets.sort_values('support', ascending=False).head()

# %% [markdown]

 ### 5.1 Add extra quality metrics
 * **Confidence / lift** via `association_rules`.
 * Custom **importance** score = *support × lift* (feel free to tweak).


In [27]:
# # %%

# rules = association_rules(freq_itemsets, metric='confidence', min_threshold=0.0)
# rules['importance'] = rules['support'] * rules['lift']
# rules.sort_values('importance', ascending=False).head()

# %% [markdown]

 ### 5.2 Store a tidy table of pattern → score


In [28]:
# # %%

# pattern_scores = (
#     rules[['antecedents', 'consequents', 'importance']]
#       .sort_values('importance', ascending=False)
#       .reset_index(drop=True)
# )

# pattern_scores.head()

# %% [markdown]

 ## 6 User–item interaction matrix for collaborative filtering
 Binary purchase indicator is enough for implicit feedback.


In [29]:
# %%

user_item = (
    train_raw
      .groupby(['User_id', 'itemDescription'])['itemDescription']
      .count()
      .unstack(fill_value=0)
      .astype(bool)
      .astype(int)
)

print('Users:', user_item.shape[0], '| Items:', user_item.shape[1])

Users: 3493 | Items: 167


In [30]:
user_item

itemDescription,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
User_id,,,,,,,,,,,,,,,,,,,,,
1000,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1003,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


# %% [markdown]

 ### 6.1 Compute cosine similarity between users


In [31]:
# %%

user_sim = cosine_similarity(user_item)
user_sim = pd.DataFrame(user_sim,
                        index=user_item.index,
                        columns=user_item.index)

In [32]:
user_sim

User_id,1000,1001,1002,1003,1004,1005,1006,1009,1010,1011,...,4988,4989,4990,4991,4992,4993,4995,4997,4999,5000
User_id,,,,,,,,,,,,,,,,,,,,,
1000,1.000000,0.258199,0.288675,0.000000,0.298142,0.000000,0.218218,0.0,0.000000,0.235702,...,0.000000,0.000000,0.0,0.204124,0.000000,0.000000,0.000000,0.408248,0.000000,0.000000
1001,0.258199,1.000000,0.223607,0.200000,0.230940,0.258199,0.338062,0.0,0.158114,0.182574,...,0.000000,0.316228,0.0,0.158114,0.258199,0.182574,0.000000,0.316228,0.000000,0.000000
1002,0.288675,0.223607,1.000000,0.000000,0.258199,0.000000,0.188982,0.0,0.000000,0.000000,...,0.447214,0.353553,0.0,0.353553,0.000000,0.000000,0.000000,0.353553,0.250000,0.250000
1003,0.000000,0.200000,0.000000,1.000000,0.230940,0.258199,0.169031,0.0,0.000000,0.182574,...,0.000000,0.316228,0.0,0.158114,0.000000,0.365148,0.000000,0.000000,0.223607,0.000000
1004,0.298142,0.230940,0.258199,0.230940,1.000000,0.149071,0.292770,0.0,0.091287,0.210819,...,0.000000,0.182574,0.0,0.365148,0.149071,0.316228,0.129099,0.365148,0.129099,0.129099
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,0.000000,0.182574,0.000000,0.365148,0.316228,0.235702,0.308607,0.0,0.000000,0.166667,...,0.000000,0.288675,0.0,0.144338,0.000000,1.000000,0.000000,0.000000,0.204124,0.204124
4995,0.000000,0.000000,0.000000,0.000000,0.129099,0.000000,0.188982,0.0,0.000000,0.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
4997,0.408248,0.316228,0.353553,0.000000,0.365148,0.000000,0.267261,0.0,0.000000,0.000000,...,0.000000,0.000000,0.0,0.250000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000


# %% [markdown]

 ## 7 Predict top‑N items for each user (without frequent itemsets)
 *Score = weighted sum of neighbours' purchase indicators.*


In [33]:
# %%

scores_cf = user_sim.dot(user_item)           # (users × items)
scores_cf = scores_cf / user_sim.sum(axis=1).values.reshape(-1,1)
scores_cf.head()

itemDescription,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
User_id,,,,,,,,,,,,,,,,,,,,,
1000,0.007913,0.043947,0.003318,0.005574,0.000000,0.000000,0.014528,0.004478,0.051063,0.040892,...,0.005700,0.004160,0.059432,0.095897,0.002565,0.067040,0.021753,0.729488,0.161202,0.010819
1001,0.010379,0.048083,0.004324,0.004588,0.000602,0.000322,0.017822,0.004678,0.045942,0.028625,...,0.007603,0.006667,0.050558,0.098863,0.001805,0.060047,0.025006,0.512302,0.166816,0.008057
1002,0.007957,0.047398,0.003977,0.004303,0.000000,0.000394,0.017532,0.003914,0.048916,0.034616,...,0.006597,0.006660,0.055352,0.095056,0.002022,0.063317,0.024205,0.574609,0.174813,0.008454
1003,0.012331,0.047645,0.005762,0.003521,0.001141,0.001220,0.024429,0.002339,0.052098,0.033797,...,0.009756,0.005793,0.042258,0.093818,0.001990,0.061429,0.028768,0.258312,0.168886,0.007056
1004,0.009758,0.047974,0.004561,0.004600,0.000316,0.000507,0.019496,0.003484,0.047972,0.037208,...,0.008367,0.006369,0.049134,0.095641,0.002237,0.060488,0.025546,0.394304,0.171617,0.006576


# %% [markdown]

 ### 7.1 Mask already‑purchased items


In [34]:
# %%

already_owned = user_item.astype(bool)
# scores_cf = scores_cf.mask(already_owned, other=np.NINF)

In [35]:
scores_cf

itemDescription,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
User_id,,,,,,,,,,,,,,,,,,,,,
1000,0.007913,0.043947,0.003318,0.005574,0.000000,0.000000,0.014528,0.004478,0.051063,0.040892,...,0.005700,0.004160,0.059432,0.095897,0.002565,0.067040,0.021753,0.729488,0.161202,0.010819
1001,0.010379,0.048083,0.004324,0.004588,0.000602,0.000322,0.017822,0.004678,0.045942,0.028625,...,0.007603,0.006667,0.050558,0.098863,0.001805,0.060047,0.025006,0.512302,0.166816,0.008057
1002,0.007957,0.047398,0.003977,0.004303,0.000000,0.000394,0.017532,0.003914,0.048916,0.034616,...,0.006597,0.006660,0.055352,0.095056,0.002022,0.063317,0.024205,0.574609,0.174813,0.008454
1003,0.012331,0.047645,0.005762,0.003521,0.001141,0.001220,0.024429,0.002339,0.052098,0.033797,...,0.009756,0.005793,0.042258,0.093818,0.001990,0.061429,0.028768,0.258312,0.168886,0.007056
1004,0.009758,0.047974,0.004561,0.004600,0.000316,0.000507,0.019496,0.003484,0.047972,0.037208,...,0.008367,0.006369,0.049134,0.095641,0.002237,0.060488,0.025546,0.394304,0.171617,0.006576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,0.008608,0.049084,0.012211,0.004448,0.000920,0.000984,0.019433,0.002828,0.046947,0.034096,...,0.010325,0.007056,0.041643,0.097773,0.000759,0.058959,0.024647,0.263266,0.170979,0.005658
4995,0.007291,0.053366,0.000000,0.001414,0.000000,0.000000,0.010379,0.001732,0.039111,0.044396,...,0.012359,0.006038,0.060072,0.086152,0.003762,0.062331,0.020412,0.287247,0.164239,0.009858
4997,0.009178,0.045594,0.003688,0.007297,0.000000,0.000000,0.015731,0.003889,0.047898,0.035245,...,0.003768,0.004263,0.059146,0.100625,0.002961,0.070128,0.025360,0.833199,0.170948,0.007849


# %% [markdown]

 ### 7.2 Top‑5 recommendations per user


In [36]:
# %%

TopN = 5
top5_cf = (
    scores_cf
      .apply(lambda row: row.nlargest(TopN).index.tolist(), axis=1)
)

top5_cf.head()

User_id
1000    [whole milk, pastry, rolls/buns, other vegetab...
1001    [whole milk, rolls/buns, soda, other vegetable...
1002    [whole milk, other vegetables, rolls/buns, sod...
1003    [rolls/buns, root vegetables, whole milk, othe...
1004    [whole milk, other vegetables, rolls/buns, roo...
dtype: object

# %% [markdown]

 ## 8 Merge CF with frequent‑pattern recommendations (optional)
 **Heuristic**
 1. For the target user, find patterns whose antecedent ⊆ their basket history.
 2. Add consequent items (not yet purchased) with weight = pattern importance.
 3. Combine CF score + pattern weight (e.g. `score_final = α·CF + (1‑α)·FP`).
 4. Rank and pick top 5.

 *Tune `alpha` on validation data.*


In [37]:
pattern_scores = pd.read_excel('top_rules.xlsx')
pattern_scores

,Unnamed: 0,antecedent,consequent,support,confidence,lift
0,0,"['whipped/sour cream', 'frankfurter']","['dishes', 'citrus fruit']",0.000239,0.500000,2090.250000
1,1,"['dishes', 'citrus fruit']","['whipped/sour cream', 'frankfurter']",0.000239,1.000000,2090.250000
2,2,"['whipped/sour cream', 'dishes']","['frankfurter', 'citrus fruit']",0.000239,0.500000,1393.500000
3,3,"['frankfurter', 'citrus fruit']","['whipped/sour cream', 'dishes']",0.000239,0.666667,1393.500000
4,4,"['dishes', 'frankfurter']","['whipped/sour cream', 'citrus fruit']",0.000239,0.666667,428.769231
...,...,...,...,...,...,...
75,75,"['white bread', 'waffles']",['other vegetables'],0.000239,0.500000,4.686659
76,76,"['softener', 'soda']",['whole milk'],0.000239,0.500000,3.831806
77,77,"['long life bakery product', 'UHT-milk']",['whole milk'],0.000239,0.500000,3.831806
78,78,"['cake bar', 'sausage']",['whole milk'],0.000239,0.500000,3.831806


In [38]:
# %%
import ast
alpha = 0.7   # 70% CF, 30% pattern‑based – tweak later

# Build a dict for fast user history lookup
user_history = train_raw.groupby('User_id')['itemDescription'].apply(set)

# Example for one user (feel free to loop later)
example_user = user_item.index[1]
history      = user_history[example_user]

# pattern candidates
candidates = []
for _, row in pattern_scores.iterrows():
    if set(ast.literal_eval(row['antecedent'])).issubset(history):
        for itm in set(ast.literal_eval(row['consequent'])):
            if itm not in history:
                candidates.append((itm,  row['lift']))
                
fp_df = pd.DataFrame(candidates, columns=['item', 'fp_score']).groupby('item').max()

# CF part for this user
cf_scores_u = scores_cf.loc[example_user].dropna().to_frame('cf_score')

merged = cf_scores_u.join(fp_df, how='outer').fillna(0)
merged['final'] = alpha*merged['cf_score'] + (1-alpha)*merged['fp_score']
merged.sort_values('final', ascending=False).head(TopN)

,cf_score,fp_score,final
yogurt,0.166816,8.484018,2.661977
whole milk,0.512302,0.000000,0.358611
rolls/buns,0.436915,0.000000,0.305840
soda,0.411241,0.000000,0.287868
other vegetables,0.218862,0.000000,0.153204


In [39]:
print(user_history)
print(example_user)
print(history)

User_id
1000                    {pastry, whole milk, salty snack}
1001    {whole milk, sausage, rolls/buns, soda, frankf...
1002    {frozen vegetables, whole milk, butter, other ...
1003    {dental care, root vegetables, rolls/buns, fro...
1004    {pastry, shopping bags, red/blush wine, frozen...
                              ...                        
4993    {shopping bags, abrasive cleaner, onions, roll...
4995      {rice, dish cleaner, mayonnaise, domestic eggs}
4997                            {whole milk, canned beer}
4999    {newspapers, semi-finished bread, detergent, o...
5000    {fruit/vegetable juice, bottled beer, onions, ...
Name: itemDescription, Length: 3493, dtype: object
1001
{'whole milk', 'sausage', 'rolls/buns', 'soda', 'frankfurter'}


# %% [markdown]

 ## 9 (Optional) Recency boost
 If user's last purchase date for an itemset is recent, we can up‑weight it.


In [40]:
# %%

# Build a simple recency factor (days since last purchase)
train_raw['Date_dt'] = pd.to_datetime(train_raw['Date'], format='%d/%m/%Y')
last_purchase = train_raw.groupby('User_id')['Date_dt'].max()

def recency_weight(u):
    today = last_purchase.max()
    delta = (today - last_purchase[u]).days + 1
    return 1 / math.log1p(delta)
example_user=1001
rec_w = recency_weight(example_user)
print('Recency weight for user', example_user, ':', rec_w)

Recency weight for user 1001 : 1.4426950408889634


In [41]:
# Modify the collaborative filtering scores with recency weight
def apply_recency_to_user(user_id):
    # Get the recency weight for this user
    rec_w = recency_weight(user_id)
    
    # Adjust the CF scores
    cf_row = scores_cf.loc[user_id].copy()
    
    # Higher recency weight = more emphasis on their recent preferences
    cf_row = cf_row * rec_w
    
    return cf_row

# Apply to all users
recency_boosted_scores = pd.DataFrame(index=scores_cf.index, columns=scores_cf.columns)
for user in scores_cf.index:
    try:
        recency_boosted_scores.loc[user] = apply_recency_to_user(user)
    except KeyError:
        # User not in last_purchase (possibly a new user)
        recency_boosted_scores.loc[user] = scores_cf.loc[user]

In [42]:
# Combine with frequent pattern mining
alpha = 0.7   # Weight for CF component
beta = 0.5    # Weight for recency effect

# For a specific user
def get_hybrid_scores_with_recency(user_id):
    if user_id not in user_item.index:
        return None
    
    # Get CF scores with recency boost
    cf_row = recency_boosted_scores.loc[user_id].dropna()
    
    # Mask already purchased items
    cf_row = cf_row.mask(user_item.loc[user_id].astype(bool), other=np.NINF)
    
    # Get pattern-based candidates
    history = user_history[user_id]
    candidates = []
    for _, row in pattern_scores.iterrows():
        antecedent = ast.literal_eval(row['antecedent'])
        if set(antecedent).issubset(history):
            for item in set(ast.literal_eval(row['consequent'])):
                if item not in history:
                    candidates.append((item, row['lift']))
    
    fp_df = pd.DataFrame(candidates, columns=['item', 'fp_score']).groupby('item').max()
    
    # Merge scores
    merged = cf_row.to_frame('cf_score').join(fp_df, how='outer').fillna(0)
    merged['final'] = alpha * merged['cf_score'] + (1-alpha) * merged['fp_score']
    
    return merged.sort_values('final', ascending=False)

In [43]:
# Generate recommendations for all users
top_n_recommendations = {}
for user in user_item.index:
    scores = get_hybrid_scores_with_recency(user)
    if scores is not None:
        top_n_recommendations[user] = scores.head(5).index.tolist()

In [44]:
top_n_recommendations

{1000: ['rolls/buns', 'other vegetables', 'soda', 'yogurt', 'bottled water'],
 1001: ['yogurt',
  'other vegetables',
  'bottled water',
  'root vegetables',
  'shopping bags'],
 1002: ['rolls/buns', 'soda', 'yogurt', 'bottled water', 'root vegetables'],
 1003: ['whole milk', 'other vegetables', 'soda', 'yogurt', 'shopping bags'],
 1004: ['soda',
  'yogurt',
  'bottled water',
  'bottled beer',
  'whipped/sour cream'],
 1005: ['whole milk', 'other vegetables', 'soda', 'yogurt', 'root vegetables'],
 1006: ['other vegetables',
  'soda',
  'yogurt',
  'root vegetables',
  'tropical fruit'],
 1009: ['whole milk',
  'other vegetables',
  'rolls/buns',
  'soda',
  'shopping bags'],
 1010: ['whole milk', 'other vegetables', 'soda', 'rolls/buns', 'yogurt'],
 1011: ['whole milk', 'other vegetables', 'soda', 'yogurt', 'root vegetables'],
 1012: ['whole milk', 'other vegetables', 'rolls/buns', 'soda', 'yogurt'],
 1013: ['rolls/buns', 'soda', 'yogurt', 'shopping bags', 'pastry'],
 1014: ['other ve

In [ ]:
#optional
def recency_item_boost(user_id):
    # Get user's purchase history with dates
    user_purchases = train_raw[train_raw['User_id'] == user_id]
    
    # Get the most recent purchase date for each item
    latest_purchase_dates = user_purchases.groupby('itemDescription')['Date_dt'].max()
    
    # Calculate days since last purchase for each item
    today = train_raw['Date_dt'].max()
    days_since_purchase = (today - latest_purchase_dates).dt.days + 1
    
    # Calculate recency weights for items
    item_recency_weights = 1 / np.log1p(days_since_purchase)
    
    return item_recency_weights

# Use this to boost recommendations of items similar to recently purchased ones

# %% [markdown]

 ## 10 Mini CLI for manual testing


In [44]:
# %%

while True:
    try:
        uid = int(input('Enter user_id (or 0 to quit): '))
    except ValueError:
        print('Not a number.')
        continue
    if uid == 0:
        break
    mode = input('Type "with" for CF+patterns, anything else for CF only: ')
    
    if uid not in user_item.index:
        print('Cold‑start user – recommend top popular items')
        popular = train_raw['itemDescription'].value_counts().head(TopN).index.tolist()
        print(popular)
        continue
    
    cf_row   = scores_cf.loc[uid].dropna()
    cf_row   = cf_row.mask(user_item.loc[uid].astype(bool), other=np.NINF)
    
    if mode.strip().lower() != 'with':
        print(cf_row.nlargest(TopN).index.tolist())
        continue
    
    hist     = user_history[uid]
    cand     = []
    for _, r in pattern_scores.iterrows():
        if set(ast.literal_eval(row['antecedent'])).issubset(hist):
            for it in set(ast.literal_eval(row['consequent'])):
                if it not in hist:
                    cand.append((it, r['lift']))
    fp_part = pd.DataFrame(cand, columns=['item', 'fp_score']).groupby('item').max()
    merged  = cf_row.to_frame('cf_score').join(fp_part, how='outer').fillna(0)
    merged['final'] = alpha*merged['cf_score'] + (1-alpha)*merged['fp_score']
    print(merged.sort_values('final', ascending=False).head(TopN).index.tolist())

['rolls/buns', 'other vegetables', 'soda', 'yogurt', 'bottled water']
['whole milk', 'other vegetables', 'rolls/buns', 'soda', 'yogurt']
['rolls/buns', 'other vegetables', 'soda', 'yogurt', 'bottled water']
Cold‑start user – recommend top popular items
['whole milk', 'other vegetables', 'rolls/buns', 'soda', 'yogurt']
Not a number.
Not a number.
Not a number.
Not a number.
Not a number.
Not a number.
Not a number.


KeyboardInterrupt: Interrupted by user

I'll analyze the provided notebook which details a grocery recommendation system implementation. The notebook covers two main approaches:

1. Mining frequent itemsets (association rules)
2. Building a collaborative filtering recommender

Let me explain how to evaluate these recommendation systems.

## How to Evaluate the Recommendations

The notebook shows two types of recommendation approaches, but lacks formalized evaluation. Here's how you could evaluate them:

### 1. Offline Evaluation Metrics

You should implement these metrics to systematically evaluate recommendation quality:

**Accuracy Metrics:**
- **Hit Rate**: Percentage of users for whom at least one recommended item appears in their test set
- **Precision@N**: Proportion of recommended items that are relevant
- **Recall@N**: Proportion of relevant items that are recommended
- **F1 Score**: Harmonic mean of precision and recall
- **NDCG (Normalized Discounted Cumulative Gain)**: Measures ranking quality, giving higher weight to items at top positions

Here's how you could implement hit rate:

```python
def calculate_hit_rate(test_data, recommendations, user_col='User_id', item_col='itemDescription'):
    """
    Calculate hit rate: percentage of users for whom at least one recommended item is in their test set
    
    Parameters:
    -----------
    test_data : DataFrame
        DataFrame containing test purchases
    recommendations : dict
        Dictionary with user_id as key and list of recommended items as value
    user_col : str
        Column name for user IDs
    item_col : str
        Column name for item descriptions
    
    Returns:
    --------
    float
        Hit rate (0 to 1)
    """
    hits = 0
    users = test_data[user_col].unique()
    
    # For each user in test data
    for user in users:
        if user not in recommendations:
            continue
            
        # Get user's actual purchases in test set
        actual_items = set(test_data[test_data[user_col] == user][item_col])
        
        # Get recommended items for this user
        recommended_items = set(recommendations[user])
        
        # Check if there's at least one hit
        if len(actual_items.intersection(recommended_items)) > 0:
            hits += 1
    
    return hits / len(users)
```

### 2. Catalog Coverage

Evaluate how much of your product catalog appears in recommendations:

```python
def calculate_coverage(recommendations, all_items):
    """
    Calculate catalog coverage: percentage of items that appear in at least one recommendation
    
    Parameters:
    -----------
    recommendations : dict
        Dictionary with user_id as key and list of recommended items as value
    all_items : set
        Set of all possible items
        
    Returns:
    --------
    float
        Coverage (0 to 1)
    """
    recommended_items = set()
    
    for user, items in recommendations.items():
        recommended_items.update(items)
    
    return len(recommended_items) / len(all_items)
```

### 3. Diversity

Measure how diverse recommendations are for each user:

```python
def calculate_diversity(recommendations):
    """
    Calculate average intra-list diversity of recommendations
    
    Parameters:
    -----------
    recommendations : dict
        Dictionary with user_id as key and list of recommended items as value
    
    Returns:
    --------
    float
        Average diversity score (0 to 1)
    """
    total_diversity = 0
    
    for user, items in recommendations.items():
        if len(items) <= 1:
            continue
            
        # Calculate all pairs of items
        pairs = [(items[i], items[j]) 
                for i in range(len(items)) 
                for j in range(i+1, len(items))]
        
        # Calculate diversity (assuming you have an item_similarity matrix)
        diversity_score = 1 - np.mean([item_similarity[pair[0]][pair[1]] for pair in pairs])
        total_diversity += diversity_score
    
    return total_diversity / len(recommendations)
```

### 4. A/B Testing Framework

For real-world evaluation, implement A/B testing:

```python
def ab_testing_setup(users, test_ratio=0.5):
    """
    Split users for A/B testing
    
    Parameters:
    -----------
    users : array-like
        List of user IDs
    test_ratio : float
        Proportion of users to assign to test group
        
    Returns:
    --------
    dict
        Dictionary with 'control' and 'test' keys containing user IDs
    """
    np.random.shuffle(users)
    split_point = int(len(users) * (1 - test_ratio))
    
    return {
        'control': users[:split_point],
        'test': users[split_point:]
    }
```

### 5. Temporal Evaluation

Since this is grocery data, evaluate performance over time to measure seasonality effects:

```python
def temporal_evaluation(test_data, recommendations, user_col='User_id', 
                         item_col='itemDescription', date_col='Date'):
    """
    Evaluate recommendations across different time periods
    
    Parameters:
    -----------
    test_data : DataFrame
        DataFrame containing test purchases with dates
    recommendations : dict
        Dictionary with user_id as key and list of recommended items as value
    user_col, item_col, date_col : str
        Column names
        
    Returns:
    --------
    dict
        Dictionary with time period as key and hit rate as value
    """
    # Convert date column to datetime if needed
    if not pd.api.types.is_datetime64_dtype(test_data[date_col]):
        test_data[date_col] = pd.to_datetime(test_data[date_col])
    
    # Group by month
    test_data['month'] = test_data[date_col].dt.month
    
    results = {}
    for month, month_data in test_data.groupby('month'):
        month_users = month_data[user_col].unique()
        month_hits = 0
        
        for user in month_users:
            if user not in recommendations:
                continue
                
            actual_items = set(month_data[month_data[user_col] == user][item_col])
            recommended_items = set(recommendations[user])
            
            if len(actual_items.intersection(recommended_items)) > 0:
                month_hits += 1
                
        results[month] = month_hits / len(month_users) if len(month_users) > 0 else 0
        
    return results
```

### 6. Hybrid vs. Individual Approach Comparison

The notebook implements a hybrid approach (combining CF and frequent pattern mining). Compare performance:

```python
# Get recommendations using pure CF
cf_recommendations = {}
for user in user_item.index:
    cf_row = scores_cf.loc[user].dropna()
    cf_row = cf_row.mask(user_item.loc[user].astype(bool), other=np.NINF)
    cf_recommendations[user] = cf_row.nlargest(TopN).index.tolist()

# Get recommendations using hybrid approach
hybrid_recommendations = {}
# Implementation similar to the code in cell [40]

# Compare metrics
cf_hit_rate = calculate_hit_rate(test_raw, cf_recommendations)
hybrid_hit_rate = calculate_hit_rate(test_raw, hybrid_recommendations)

print(f"Pure CF Hit Rate: {cf_hit_rate:.4f}")
print(f"Hybrid Hit Rate: {hybrid_hit_rate:.4f}")
```

### 7. Parameter Sensitivity Analysis

Test different parameter settings to find optimal values:

```python
# Test different alpha values (weight between CF and pattern mining)
alpha_values = [0.1, 0.3, 0.5, 0.7, 0.9]
results = {}

for alpha in alpha_values:
    hybrid_recommendations = {}
    # Implementation with current alpha value
    
    results[alpha] = calculate_hit_rate(test_raw, hybrid_recommendations)

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(list(results.keys()), list(results.values()), marker='o')
plt.xlabel('Alpha Value')
plt.ylabel('Hit Rate')
plt.title('Performance with Different Alpha Values')
plt.grid(True)
plt.show()
```

### 8. Cold-Start User Evaluation

Evaluate performance specifically for cold-start users (those with few or no prior purchases):

```python
def evaluate_cold_start(train_data, test_data, recommendations, 
                       user_col='User_id', item_col='itemDescription', 
                       purchase_threshold=5):
    """
    Evaluate recommendation performance for cold-start users
    
    Parameters:
    -----------
    train_data, test_data : DataFrame
        DataFrames containing train and test purchases
    recommendations : dict
        Dictionary with user_id as key and list of recommended items as value
    user_col, item_col : str
        Column names
    purchase_threshold : int
        Maximum number of purchases to be considered a cold-start user
        
    Returns:
    --------
    dict
        Dictionary with metrics for cold-start and regular users
    """
    # Count purchases per user in training set
    user_purchase_counts = train_data.groupby(user_col).size()
    
    # Identify cold-start users
    cold_start_users = user_purchase_counts[user_purchase_counts <= purchase_threshold].index
    regular_users = user_purchase_counts[user_purchase_counts > purchase_threshold].index
    
    # Calculate metrics for each group
    cold_start_hit_rate = calculate_hit_rate(
        test_data[test_data[user_col].isin(cold_start_users)], 
        recommendations
    )
    
    regular_hit_rate = calculate_hit_rate(
        test_data[test_data[user_col].isin(regular_users)], 
        recommendations
    )
    
    return {
        'cold_start_hit_rate': cold_start_hit_rate,
        'regular_hit_rate': regular_hit_rate,
        'cold_start_count': len(cold_start_users),
        'regular_count': len(regular_users)
    }
```

## Implementation Steps

To properly evaluate your grocery recommendation system:

1. **Split the data properly**: Make sure your test set is from a later time period than your training set
2. **Generate recommendations**: Apply both methods to all users in test set
3. **Calculate metrics**: Implement the metrics described above
4. **Visualize results**: Create charts comparing different approaches
5. **Tune parameters**: Optimize parameters like minimum support threshold and alpha

This approach will give you a comprehensive view of your recommendation system's performance and help identify which method works best for your grocery dataset.